In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ----- Dataset -----
np.random.seed(0)
X = np.random.randn(200, 2)
X

array([[ 1.76405235,  0.40015721],
       [ 0.97873798,  2.2408932 ],
       [ 1.86755799, -0.97727788],
       [ 0.95008842, -0.15135721],
       [-0.10321885,  0.4105985 ],
       [ 0.14404357,  1.45427351],
       [ 0.76103773,  0.12167502],
       [ 0.44386323,  0.33367433],
       [ 1.49407907, -0.20515826],
       [ 0.3130677 , -0.85409574],
       [-2.55298982,  0.6536186 ],
       [ 0.8644362 , -0.74216502],
       [ 2.26975462, -1.45436567],
       [ 0.04575852, -0.18718385],
       [ 1.53277921,  1.46935877],
       [ 0.15494743,  0.37816252],
       [-0.88778575, -1.98079647],
       [-0.34791215,  0.15634897],
       [ 1.23029068,  1.20237985],
       [-0.38732682, -0.30230275],
       [-1.04855297, -1.42001794],
       [-1.70627019,  1.9507754 ],
       [-0.50965218, -0.4380743 ],
       [-1.25279536,  0.77749036],
       [-1.61389785, -0.21274028],
       [-0.89546656,  0.3869025 ],
       [-0.51080514, -1.18063218],
       [-0.02818223,  0.42833187],
       [ 0.06651722,

In [ ]:
y = (X[:, 0] + X[:, 1] > 0).astype(int).reshape(-1, 1)
n, d = X.shape

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# ----- Logistic Regression -----
def train_lr(X, y, lr=0.1, epochs=2000):
    n, d = X.shape
    W = np.zeros((d, 1))
    b = 0.0
    for _ in range(epochs):
        z = X @ W + b
        p = sigmoid(z)
        dW = (1/n) * X.T @ (p - y)
        db = (1/n) * np.sum(p - y)
        W -= lr * dW
        b -= lr * db
    return W, b

W_lr, b_lr = train_lr(X, y)

# ----- Manual Single Neuron NN -----
def train_nn(X, y, lr=0.1, epochs=2000):
    n, d = X.shape
    W = np.zeros((d, 1))
    b = 0.0
    for _ in range(epochs):
        z = X @ W + b
        a = sigmoid(z)
        dW = (1/n) * X.T @ (a - y)
        db = (1/n) * np.sum(a - y)
        W -= lr * dW
        b -= lr * db
    return W, b

W_nn, b_nn = train_nn(X, y)

In [6]:
# ----- PyTorch Neuron -----
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(2, 2)
        self.l2 = nn.Linear(2, 1)
    def forward(self, x):
        z = self.l1(x)
        zp = torch.sigmoid(z)
        y = self.l2(zp)
        yp = torch.sigmoid(y)
        return yp

torch.manual_seed(0)
model = SimpleNN()
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

for _ in range(2000):
    optimizer.zero_grad()
    pred = model(X_t)
    loss = criterion(pred, y_t)
    loss.backward()
    # print(loss)
    optimizer.step()

y_test = model(X_t)
y_test

tensor([[0.9940],
        [0.9941],
        [0.9879],
        [0.9859],
        [0.9048],
        [0.9936],
        [0.9884],
        [0.9856],
        [0.9927],
        [0.0278],
        [0.0057],
        [0.6845],
        [0.9854],
        [0.2456],
        [0.9941],
        [0.9684],
        [0.0055],
        [0.1941],
        [0.9941],
        [0.0173],
        [0.0055],
        [0.8906],
        [0.0097],
        [0.0443],
        [0.0057],
        [0.0363],
        [0.0058],
        [0.9422],
        [0.9310],
        [0.0090],
        [0.0086],
        [0.0055],
        [0.1482],
        [0.0076],
        [0.0117],
        [0.9878],
        [0.2741],
        [0.1017],
        [0.0063],
        [0.1321],
        [0.1416],
        [0.0079],
        [0.9941],
        [0.9903],
        [0.5145],
        [0.9875],
        [0.9923],
        [0.9913],
        [0.9939],
        [0.9680],
        [0.9625],
        [0.1155],
        [0.9867],
        [0.0074],
        [0.9941],
        [0

In [3]:
W_t = model.l1.weight.detach().numpy().reshape(-1, 1)
b_t = model.l1.bias.detach().numpy()[0]

# ----- Grid for decision boundary -----
xs = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200)
ys = np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200)
xx, yy = np.meshgrid(xs, ys)
grid = np.c_[xx.ravel(), yy.ravel()]




def decision_boundary(W, b):
    z = grid @ W + b
    p = sigmoid(z)
    return p.reshape(xx.shape)

Z_lr = decision_boundary(W_lr, b_lr)
Z_nn = decision_boundary(W_nn, b_nn)
Z_t  = decision_boundary(W_t,  b_t)

# ----- Plot 1: Logistic Regression -----
plt.figure(figsize=(5, 5))
plt.contour(xx, yy, Z_lr, levels=[0.5])
plt.scatter(X[:, 0], X[:, 1], c=y.ravel())
plt.title("Logistic Regression Decision Boundary")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

# ----- Plot 2: Manual Neural Neuron -----
plt.figure(figsize=(5, 5))
plt.contour(xx, yy, Z_nn, levels=[0.5])
plt.scatter(X[:, 0], X[:, 1], c=y.ravel())
plt.title("Manual Single-Neuron NN Decision Boundary")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

# ----- Plot 3: PyTorch Neuron -----
plt.figure(figsize=(5, 5))
plt.contour(xx, yy, Z_t, levels=[0.5])
plt.scatter(X[:, 0], X[:, 1], c=y.ravel())
plt.title("PyTorch Single Neuron Decision Boundary")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 4 is different from 2)